# Jon2 AI — Kaggle Compute and Engine Test

فحص الموارد ثم اختبار بنية Transformer الأصلية وفحص بيانات التدريب. لا يبدأ هذا الدفتر تدريبًا أو يحمّل نموذجًا خارجيًا.

In [ ]:
import platform, sys
print('JON2 AI COMPUTE TEST')
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import psutil
    memory = psutil.virtual_memory()
    print(f'RAM total: {memory.total / (1024**3):.2f} GB')
    print(f'RAM available: {memory.available / (1024**3):.2f} GB')
except Exception as exc:
    print('RAM check unavailable:', type(exc).__name__)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'GPU memory: {props.total_memory / (1024**3):.2f} GB')
    probe = torch.randn((1024, 1024), device='cuda')
    print('GPU tensor test:', float((probe @ probe).mean().item()))
else:
    print('GPU unavailable in this Kaggle session; CPU tests will continue.')


## اختبار ملفات المشروع ومحرك Jon2 الأصلي

In [ ]:
from pathlib import Path
import subprocess, sys
project_dir = Path('/kaggle/working/jon2-ai')
if not project_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/jonkabeel-star/jon2-ai.git', str(project_dir)], check=True)
engine_dir = project_dir / 'ai_engine'
print('Project checkout:', project_dir)
audit = subprocess.run([sys.executable, str(engine_dir / 'audit_data.py')], text=True, capture_output=True)
print('--- Training data audit ---')
print(audit.stdout)
if audit.stderr:
    print('Audit notes:', audit.stderr[:1500])
print('Data audit exit code:', audit.returncode)
sys.path.insert(0, str(engine_dir))
from jon2_core import Jon2Transformer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Jon2Transformer(vocab_size=64, width=32, heads=4, layers=1, context=32, dropout=0.0).to(device)
sample = torch.randint(0, 64, (2, 16), device=device)
logits = model(sample)
assert tuple(logits.shape) == (2, 16, 64), f'Unexpected output shape: {tuple(logits.shape)}'
loss = logits.float().mean()
loss.backward()
assert any(parameter.grad is not None for parameter in model.parameters()), 'No gradients were produced.'
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print('Original Jon2 Transformer forward pass: PASS')
print('Original Jon2 Transformer backward pass: PASS')
print('Device used:', device)
print('Smoke-test parameters:', f'{parameter_count:,}')
print('IMPORTANT: this is an architecture smoke test with temporary random weights, not a trained assistant.')


## بوابة التدريب

لا يبدأ التدريب إلا بعد مراجعة واعتماد حقوق البيانات وتجهيز مجموعة عربية أصلية أو مرخصة بالحجم والجودة المناسبين. ملفات البذور الحالية للاختبار فقط.